# _TITLE_

Domain Explanation and Background

## Setup

Importing libraries


In [ ]:
import io
import os
import pathlib
from collections.abc import Callable, Collection
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import chain
from typing import NamedTuple

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from pydantic import BaseModel, ConfigDict, Field, ValidationError


In [2]:
np.random.default_rng(2929)

Generator(PCG64) at 0xFF3AEA6E2C00

In [ ]:
cwd = pathlib.Path(os.getcwd())

data_dir = cwd / "data"

raw_data_dir = data_dir / "raw"
interim_data_dir = data_dir / "interim"
processed_data_dir = data_dir / "processed"

In [3]:
def preview(df: pd.DataFrame, n: int = 10) -> None:
    display(df.sample(n))

In [4]:
class DeepSeaCoralRecord(BaseModel):
    model_config = ConfigDict(populate_by_name=True, frozen=True)

    lat: float = Field(alias="latitude")
    lon: float = Field(alias="longitude")
    catalog_num: int = Field(alias="CatalogNumber")
    record_type: str = Field(alias="RecordType")
    loc_accuracy: str | None = Field(default=None, alias="LocationAccuracy")
    scientific_nm: str | None = Field(default=None, alias="ScientificName")
    depth: float | None = Field(default=None, alias="DepthInMeters")
    gis_depth: float | None = Field(default=None, alias="gisGEBCODepth")
    large_marine_ecosystem: str = Field(alias="LargeMarineEcosystem")
    country: str | None = Field(default=None, alias="Country")
    obs_date: str = Field(alias="ObservationDate")
    event_id: str = Field(alias="EventID")
    survey_id: str = Field(alias="SurveyID")
    aphia_id: int | None = Field(default=None, alias="AphiaID")
    genus: str | None = Field(default=None, alias="Genus")
    family: str | None = Field(default=None, alias="Family")
    order: str | None = Field(default=None, alias="Order")
    count: int | None = Field(default=None, alias="IndividualCount")
    density: float | None = Field(default=None, alias="Density")
    sample_area: float | None = Field(default=None, alias="SampleAreaInSquareMeters")
    temp: float | None = Field(default=None, alias="Temperature")
    salinity: float | None = Field(default=None, alias="Salinity")
    oxygen: float | None = Field(default=None, alias="Oxygen")
    substrate: str | None = Field(default=None, alias="Substrate")
    cmecs_substrate: str | None = Field(default=None, alias="CMECSSubstrate")
    cmecs_geo_form: str | None = Field(default=None, alias="CMECSGeoForm")
    is_shallow: bool | None = Field(default=None, alias="ShallowFlag")


class DeepSeaCoralError(NamedTuple):
    line: str
    error: ValidationError

In [5]:
deep_sea_corals_url = (
    r"https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.jsonlKVP"
    r"?CatalogNumber,RecordType,LocationAccuracy,ScientificName,latitude,longitude,DepthInMeters,LargeMarineEcosystem,ObservationDate,EventID,SurveyID,AphiaID,Genus,Family,Order,IndividualCount,Density,Temperature,Salinity,Oxygen,SampleAreaInSquareMeters,gisGEBCODepth,Country,Substrate,CMECSSubstrate,CMECSGeoForm,ShallowFlag"
    r"&ShallowFlag=0"
    r"&ObservationYear%3E=2018"
    r"&Country=%22USA%22"
    r"&LargeMarineEcosystem=~%22%28Southeast%20U.S.%20Continental%20Shelf%7CGulf%20of%20Mexico%7CCaribbean%20Sea%29%22"
)

records: list[DeepSeaCoralRecord] = []
errors: list[DeepSeaCoralError] = []

with requests.get(deep_sea_corals_url, stream=True, timeout=600) as r:
    r.raise_for_status()

    for line in r.iter_lines():
        try:
            record = DeepSeaCoralRecord.model_validate_json(line)
            records.append(record)

        except ValidationError as e:
            errors.append(DeepSeaCoralError(line=line.decode(), error=e))

df = pd.DataFrame(record.model_dump() for record in records)

total_lines = len(records) + len(errors)

print(f"Total lines processed: {total_lines}")
print(f"Valid records: {len(df)}")
print(f"Errors: {len(errors)}")

for error in errors:
    print(f"Validation error ({error.error}) for line: {error.line}")

Total lines processed: 84756
Valid records: 84756
Errors: 0


In [10]:
class BioOracleSource(NamedTuple):
    dataset_url: str
    variables: tuple[str, ...]


_BIO_ORACLE_SOURCES: list[BioOracleSource] = [
    BioOracleSource(
        "https://erddap.bio-oracle.org/erddap/griddap/thetao_baseline_2000_2019_depthmean.csv",
        ("thetao_mean",),
    ),
    BioOracleSource(
        "https://erddap.bio-oracle.org/erddap/griddap/so_baseline_2000_2019_depthmean.csv",
        ("so_mean",),
    ),
    BioOracleSource(
        "https://erddap.bio-oracle.org/erddap/griddap/o2_baseline_2000_2018_depthmean.csv",
        ("o2_mean",),
    ),
    BioOracleSource(
        "https://erddap.bio-oracle.org/erddap/griddap/ph_baseline_2000_2018_depthmean.csv",
        ("ph_mean",),
    ),
    BioOracleSource(
        "https://erddap.bio-oracle.org/erddap/griddap/no3_baseline_2000_2018_depthmean.csv",
        ("no3_mean",),
    ),
    BioOracleSource(
        "https://erddap.bio-oracle.org/erddap/griddap/po4_baseline_2000_2018_depthmean.csv",
        ("po4_mean",),
    ),
    BioOracleSource(
        "https://erddap.bio-oracle.org/erddap/griddap/chl_baseline_2000_2018_depthmean.csv",
        ("chl_mean",),
    ),
    BioOracleSource(
        "https://erddap.bio-oracle.org/erddap/griddap/sws_baseline_2000_2019_depthmean.csv",
        ("sws_mean",),
    ),
]

_TIME_START = "2000-01-01T00:00:00Z"
_TIME_END = "2010-01-01T00:00:00Z"

_IMLGS_BASE = "https://www.ncei.noaa.gov/erddap/tabledap/IMLGS_Marine_Geology.csv"
_IMLGS_MIN_DEPTH_M = 50
_IMLGS_BBOX_BUFFER_DEG = 1.0


def _validate_df(df: pd.DataFrame) -> None:
    missing = {"lat", "lon"} - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")


def _coordinate_snap(df: pd.DataFrame) -> tuple[float, float, float, float]:
    def snap(value: float, origin: float, fn: Callable[[float], float]) -> float:
        return round(fn((value - origin) / 0.05) * 0.05 + origin, 10)

    return (
        snap(df["lat"].min(), -89.975, np.floor),
        snap(df["lon"].min(), -179.975, np.floor),
        snap(df["lat"].max(), -89.975, np.ceil),
        snap(df["lon"].max(), -179.975, np.ceil),
    )


def _fetch_bio_oracle_grid(
    dataset_url: str,
    variables: Collection[str],
    lat_min: float,
    lon_min: float,
    lat_max: float,
    lon_max: float,
) -> gpd.GeoDataFrame:
    url = (
        f"{dataset_url}"
        f"?{','.join(variables)}[({_TIME_START}):({_TIME_END})]"
        f"[({lat_min}):({lat_max})][({lon_min}):({lon_max})]"
    )

    response = requests.get(url, timeout=600)
    response.raise_for_status()

    raw = pd.read_csv(io.StringIO(response.text), skiprows=[1], parse_dates=["time"])
    latest = (
        raw.dropna(subset=list(variables))
        .loc[lambda d: d["time"] == d["time"].max()]
        .drop(columns=["time"])
    )

    return gpd.GeoDataFrame(
        latest[list(variables)],
        geometry=gpd.points_from_xy(latest["longitude"], latest["latitude"]),
        crs="EPSG:4326",
    ).to_crs("EPSG:3395")


def _fetch_imlgs_grid(
    lat_min: float,
    lon_min: float,
    lat_max: float,
    lon_max: float,
) -> gpd.GeoDataFrame:
    url = (
        f"{_IMLGS_BASE}"
        f"?latitude,longitude,WATER_DEPTH,LITH1"
        f"&latitude>={lat_min - _IMLGS_BBOX_BUFFER_DEG}"
        f"&latitude<={lat_max + _IMLGS_BBOX_BUFFER_DEG}"
        f"&longitude>={lon_min - _IMLGS_BBOX_BUFFER_DEG}"
        f"&longitude<={lon_max + _IMLGS_BBOX_BUFFER_DEG}"
        f"&WATER_DEPTH>={_IMLGS_MIN_DEPTH_M}"
    )

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    raw = pd.read_csv(io.StringIO(response.text), skiprows=[1])
    samples = raw.dropna(subset=["LITH1"]).rename(columns={"LITH1": "lith1"})

    return gpd.GeoDataFrame(
        samples[["lith1"]],
        geometry=gpd.points_from_xy(samples["longitude"], samples["latitude"]),
        crs="EPSG:4326",
    ).to_crs("EPSG:3395")


def fetch_join_contextual_data(df: pd.DataFrame) -> pd.DataFrame:
    """Fetch all contextual variables in parallel, joining each as it arrives."""
    _validate_df(df)

    lat_min, lon_min, lat_max, lon_max = _coordinate_snap(df)

    result = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326",
    ).to_crs("EPSG:3395")

    with ThreadPoolExecutor() as executor:
        futures = chain(
            (
                executor.submit(
                    _fetch_bio_oracle_grid,
                    source.dataset_url,
                    source.variables,
                    lat_min,
                    lon_min,
                    lat_max,
                    lon_max,
                )
                for source in _BIO_ORACLE_SOURCES
            ),
            (
                executor.submit(
                    _fetch_imlgs_grid,
                    df["lat"].min(),
                    df["lon"].min(),
                    df["lat"].max(),
                    df["lon"].max(),
                ),
            ),
        )

        for future in as_completed(futures):
            result = gpd.sjoin_nearest(result, future.result(), how="left").drop(
                columns=["index_right"]
            )

    return pd.DataFrame(
        result.to_crs("EPSG:4326").groupby(level=0).first().drop(columns=["geometry"])
    )


In [11]:
df = fetch_join_contextual_data(df)

In [12]:
preview(df)

,lat,lon,catalog_num,record_type,loc_accuracy,scientific_nm,depth,gis_depth,large_marine_ecosystem,country,...,is_shallow,lith1,so_mean,ph_mean,o2_mean,no3_mean,chl_mean,thetao_mean,po4_mean,sws_mean
6074,29.36690,-79.72870,968491,video observation,20m,Keratoisididae,734.0,733.0,Southeast U.S. Continental Shelf,USA,...,False,"calcareous, biogenic",35.090338,7.905691,173.927944,24.200763,0.003682,8.384758,1.578622,0.185783
52528,29.43947,-87.57601,1652287,video observation,20m,Nicella,67.0,77.0,Gulf of Mexico,USA,...,False,terrigenous,36.403882,8.034090,176.205857,5.836460,0.221213,19.794005,0.414622,0.092894
62685,29.44947,-87.59133,1662444,video observation,20m,Nicella,63.0,72.0,Gulf of Mexico,USA,...,False,terrigenous,36.403882,8.034090,176.205857,5.836460,0.221213,19.794005,0.414622,0.092894
13315,18.39503,-67.65361,1045114,video observation | specimen,20m,Chrysogorgiidae,349.0,366.0,Caribbean Sea,USA,...,False,volcanics,35.989808,7.938143,158.718374,17.901384,0.004709,14.658463,1.122239,0.054239
46516,29.44191,-87.53728,1646275,video observation,20m,Stichopathes,68.0,74.0,Gulf of Mexico,USA,...,False,terrigenous,36.341822,7.963790,140.885640,11.399956,0.106624,18.177869,0.764854,0.079982
66471,29.44276,-87.55425,1666230,video observation,20m,Anthiadidae,71.0,75.0,Gulf of Mexico,USA,...,False,terrigenous,36.403882,8.034090,176.205857,5.836460,0.221213,19.794005,0.414622,0.092894
83991,29.32451,-87.77711,1683750,video observation,20m,Scleractinia,104.0,127.0,Gulf of Mexico,USA,...,False,terrigenous,36.364597,7.942151,133.062347,13.107030,0.075534,18.237656,0.870360,0.071713
57591,29.44270,-87.58619,1657350,video observation,20m,Bebryce,69.0,73.0,Gulf of Mexico,USA,...,False,terrigenous,36.403882,8.034090,176.205857,5.836460,0.221213,19.794005,0.414622,0.092894
60466,29.44611,-87.59229,1660225,video observation,20m,Stichopathes,68.0,73.0,Gulf of Mexico,USA,...,False,terrigenous,36.403882,8.034090,176.205857,5.836460,0.221213,19.794005,0.414622,0.092894
21419,30.41758,-79.22557,1071246,still image,20m,Porifera,812.0,801.0,Southeast U.S. Continental Shelf,USA,...,False,"calcareous, biogenic",35.139472,7.917494,173.379353,23.506221,0.004013,8.539704,1.531124,0.290031
